# JS09 - Lab 4
## ANNOY vs. FAISS vs. HNSW

Eksperimen ini membandingkan ketiga library pada dataset yang jauh lebih besar:
**1.000.000 vektor, 5 dimensi**. Untuk setiap library diukur *build time* dan *query time*,
lalu hasil tetangganya dibandingkan dengan hasil *exact search* (`IndexFlatL2`) untuk menghitung
**recall@10**.

Catatan: jika RAM terbatas, `n_data` dapat diturunkan menjadi 100.000 seperti disebut pada modul.

### Step 1 - Import Libraries

In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

### Step 2 - Build the Dataset

In [2]:
# ===============================
# Build 1 million rows with 5 dimensions
# ===============================
n_data = 1_000_000   # use 100_000 if memory is constrained
dim = 5
np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

print("Dataset:", X.shape, X.dtype)
print("Query  :", query)

Dataset: (1000000, 5) float32
Query  : [[0.5166533  0.9373372  0.45094103 0.67448807 0.1888479 ]]


### Step 3 - Annoy

Annoy membangun beberapa *random projection tree*. Pencarian dilakukan dengan menelusuri tree
dan mengumpulkan kandidat; `search_k` mengatur seberapa banyak kandidat yang diperiksa.

In [3]:
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.perf_counter()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time_ann = time.perf_counter() - start

start = time.perf_counter()
neighbors, dists = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time_ann = time.perf_counter() - start

print("Build time:", build_time_ann, "seconds")
print("Query time:", query_time_ann, "seconds")
print("Neighbors:", neighbors[:5], "...")

=== Annoy ===


Build time: 13.25002730000051 seconds
Query time: 0.00011180000001331791 seconds
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


### Step 4 - FAISS (Flat Index)

`IndexFlatL2` adalah pencarian **exact**: tidak ada struktur index, seluruh vektor dipindai.
Inilah yang menjadi acuan recall.

In [4]:
print("=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.perf_counter()
faiss_index.add(X)
build_time_faiss = time.perf_counter() - start

start = time.perf_counter()
distances, indices = faiss_index.search(query, k)
query_time_faiss = time.perf_counter() - start

print("Build time:", build_time_faiss, "seconds")
print("Query time:", query_time_faiss, "seconds")
print("Neighbors:", indices[0][:5], "...")

=== FAISS (IndexFlatL2) ===
Build time: 0.006258499999603373 seconds
Query time: 0.01034090000030119 seconds
Neighbors: [901095 495168 561017 163986 185734] ...


### Step 5 - HNSW (hnswlib)

HNSW membangun graf berlapis. `ef_construction` mengatur kualitas graf saat pembangunan
(proses build jauh lebih mahal daripada Annoy), sedangkan `ef` mengatur akurasi saat query.

In [5]:
print("=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.perf_counter()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time_hnsw = time.perf_counter() - start

hnsw_index.set_ef(50)

start = time.perf_counter()
labels, hnsw_dists = hnsw_index.knn_query(query, k=k)
query_time_hnsw = time.perf_counter() - start

print("Build time:", build_time_hnsw, "seconds")
print("Query time:", query_time_hnsw, "seconds")
print("Neighbors:", labels[0][:5], "...")

=== HNSW (hnswlib) ===


Build time: 46.537909100000434 seconds
Query time: 0.00017240000033780234 seconds
Neighbors: [901095 495168 561017 163986 185734] ...


### Step 6 - Compare Build and Query Time

Waktu satu kali query dipengaruhi noise pengukuran, sehingga query diulang beberapa kali
dan diambil rata-ratanya. Untuk Annoy, `get_nns_by_vector` dipanggil ulang dengan
`search_k` yang sama.

In [6]:
def bench(fn, repeat=200):
    fn()                                     # warm-up
    start = time.perf_counter()
    for _ in range(repeat):
        fn()
    return (time.perf_counter() - start) / repeat * 1000   # ms/query

q_ann = bench(lambda: ann_index.get_nns_by_vector(query[0], k, search_k=1000))
q_faiss = bench(lambda: faiss_index.search(query, k))
q_hnsw = bench(lambda: hnsw_index.knn_query(query, k=k))

print(f"{'Method':<22}{'build (s)':>12}{'query (ms)':>14}")
print("-" * 48)
print(f"{'Annoy (10 trees)':<22}{build_time_ann:>12.2f}{q_ann:>14.4f}")
print(f"{'FAISS (FlatL2)':<22}{build_time_faiss:>12.2f}{q_faiss:>14.4f}")
print(f"{'HNSW (ef=50)':<22}{build_time_hnsw:>12.2f}{q_hnsw:>14.4f}")

Method                   build (s)    query (ms)
------------------------------------------------
Annoy (10 trees)             13.25        0.0897
FAISS (FlatL2)                0.01        2.9772
HNSW (ef=50)                 46.54        0.0285


### Step 7 - Recall Against the Exact Result

FAISS `IndexFlatL2` dipakai sebagai *ground truth*. Recall@10 mengukur berapa banyak dari 10
tetangga sebenarnya yang berhasil ditemukan oleh masing-masing metode approximate.

In [7]:
def recall_at_k(pred, truth, k=10):
    return len(set(pred[:k]) & set(truth[:k])) / k

gt = indices[0]
print("Ground truth (exact, first 5):", gt[:5])
print("Annoy recall@10:", recall_at_k(np.array(neighbors), gt))
print("HNSW  recall@10:", recall_at_k(labels[0], gt))

Ground truth (exact, first 5): [901095 495168 561017 163986 185734]
Annoy recall@10: 1.0
HNSW  recall@10: 1.0


### Step 8 - Interpretation

Hasil pengukuran pada 1.000.000 vektor 5D (nilai persisnya lihat tabel di atas):

* **FAISS Flat** selalu memberi recall 1.0 karena memang *exact*, tetapi query-nya **paling lambat
  satu sampai dua orde** daripada Annoy/HNSW karena seluruh 1 juta vektor dipindai setiap query.
* **Annoy** memberi recall 1.0 dengan build paling murah di antara dua metode approximate, tetapi
  query-nya lebih lambat daripada HNSW.
* **HNSW** memberi recall 1.0 dengan query **tercepat**, tetapi build-nya paling mahal karena harus
  menyusun graf berlapis.

Pada dimensi rendah (5D) kedua metode approximate sudah **praktis exact** - recall 1.0 - sehingga
yang membedakan hanyalah biaya build dan query. Perbedaan recall baru muncul pada dimensi tinggi
atau ketika parameter pencarian diturunkan (lihat Lab 5).

Pemilihan library karena itu bergantung pada pola pemakaian: index yang sering dibangun ulang
lebih cocok dengan FAISS/Annoy, sedangkan index yang dibangun sekali lalu di-query terus-menerus
lebih cocok dengan HNSW.